In [ ]:
import numpy as np
from avapi.carla import CarlaScenesManager

data_dir = "../examples/dataset_aerial/sim_results"
CSM = CarlaScenesManager(data_dir=data_dir)
CDM = CSM.get_scene_dataset_by_index(0)

In [ ]:
objects_local[0].box

In [ ]:
from avstack.maskfilters import box_in_fov


# get data for the frame
frame_idx = 10
agent_idx = 0
frame = CDM.get_frames(sensor="camera-0", agent=0)[frame_idx]
agent = CDM.get_agents(frame=frame)[agent_idx]
agent_ref = agent.as_reference()
objects_global = CDM.get_objects_global(frame=frame, ignore_static_agents=True)
img = CDM.get_image(frame=frame, sensor="camera-0", agent=agent_idx)
depth_img = CDM.get_depth_image(frame=frame, sensor="depthcamera-0", agent=agent_idx)

objects_local = [obj.change_reference(agent_ref, inplace=False) for obj in objects_global]
objects_local = [
    obj
    for obj in objects_local
    if box_in_fov(obj.box, img.calibration, d_thresh=150, check_reference=True)
]

In [ ]:
import matplotlib.pyplot as plt

plt.hist(depth_img.depths.reshape((-1,)))
plt.show()

In [ ]:
# show the images
img.view()
depth_img.view()

In [ ]:
from avapi.visualize.snapshot import show_image_with_boxes

show_image_with_boxes(img=img, boxes=objects_local, inline=True)

In [ ]:
# check out the occlusion/depth of an object
obj = objects_local[0]
check_reference = True
depth_image = depth_img

box_2d = obj.box.project_to_2d_bbox(
    depth_image.calibration, check_reference=check_reference
).squeeze(
    depth_image.calibration.height,
    depth_image.calibration.width,
    inplace=False,
)
depths = depth_image.depths[
    int(box_2d.ymin) : int(box_2d.ymax), int(box_2d.xmin) : int(box_2d.xmax)
]
centered_depths = np.reshape(depths, -1) - (
    obj.position.norm() - obj.box.l / 2
)

In [ ]:
obj.position

In [ ]:
depths